# Zadanie 3.2: sieć rezerwuarowa (ESN) uczona na danych z PDE

Cel (issue #21): nauczyć sieć Echo State Network (`reservoirpy`) na **danych wygenerowanych przez pełny model PDE**
(tak jak „dane rzeczywiste” w narracji projektu), w przedziale uczącym, i porównać jej prognozę z ręcznie dobranym
surogatem `ODE_2`. Wejście sieci to $(M(t),\, U_{\Delta t}(t))$, wyjście to $M(t + \Delta t)$:
1. **generalizacja na nowy harmonogram leczenia:** uczenie na 7 harmonogramach, a potem pełny rollout zamknięty
   (closed-loop) od $t = 0$ na dwóch niewidzianych harmonogramach (bazowy regularny i kurs dzielony z przerwą),
2. **ekstrapolacja w czasie:** uczenie wyłącznie na $[0, T_{\text{train}}]$ i autonomiczne rozwijanie poza $T_{\text{train}}$
   (z podawaniem znanego $U(t)$ i sprzężeniem zwrotnym $\hat{M}$),
3. **dobór hiperparametrów na osobnym zbiorze walidacyjnym:** `units`, `sr`, `lr` i regularyzacja `ridge`,
4. **porównanie ESN a `ODE_2` a PDE** oraz odporność na ziarno rezerwuaru.

Zamiast próbkować wąski impuls $U(t_k)$ w jednym punkcie (co grozi aliasingiem),
na krok $[t_k, t_{k+1}]$ podajemy dokładną całkę dawki $U_{\Delta t}(t_k) = \int_{t_k}^{t_{k+1}} U(\tau)\,d\tau$
z `Schedule.integral`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from reservoirpy.nodes import Reservoir, Ridge

from offtarget.config import FIGURES, TABLES, seed_everything
from offtarget.model import NO_TREATMENT, Params, Schedule, grid, initial_condition
from offtarget.observables import min_mass, time_to_regrowth, tumor_mass
from offtarget.ode import ODE2Params, initial_beam_fraction, simulate_ode2
from offtarget.solvers import simulate

rng = seed_everything(42)

# Wspólna geometria i scenariusz referencyjny: pole przesunięte (s = 6, r = 8, beta = 1.5),
# gdzie widać zarówno spadek masy pod wiązką, jak i wyraźny odrost z części nieobjętej wiązką.
x, dx = grid(80.0, 800)
t = np.round(np.linspace(0.0, 10.0, 201), 10)  # dt = 0.05
u0 = initial_condition(x)
M0 = float(tumor_mass(u0, x))

p_ref = Params(beta=1.5, s=6.0, r=8.0)
c0_ref = initial_beam_fraction(x, u0, p_ref)
ode2_ref = ODE2Params(rho=p_ref.rho, K_eff=30.0, beta=p_ref.beta, c=c0_ref, gamma=1.2 * p_ref.D)

M_SCALE = 35.0  # ręczne przeskalowanie M(t) do przedziału rzędu [0, 1]


def dose_steps(sched: Schedule, t_grid: np.ndarray) -> np.ndarray:
    """Dokładna całka U(t) na każdym kroku [t_k, t_{k+1}] (długość len(t_grid) - 1)."""
    return np.array([sched.integral(float(t_grid[k]), float(t_grid[k + 1])) for k in range(len(t_grid) - 1)])


def pde_mass(sched: Schedule) -> np.ndarray:
    """Masa guza M(t) z pełnego PDE (solver CN): to są nasze dane „rzeczywiste”."""
    return tumor_mass(simulate(p_ref, sched, x, t, method="cn"), x)


def ode2_mass(sched: Schedule) -> np.ndarray:
    mi, mo = simulate_ode2(c0_ref * M0, (1.0 - c0_ref) * M0, ode2_ref, sched, t)
    return mi + mo


def build_xy(M_traj: np.ndarray, sched: Schedule) -> tuple[np.ndarray, np.ndarray]:
    """Tworzy pary (X_k = [M_k / scale, U_step_k], Y_k = [M_{k+1} / scale])."""
    u_step = dose_steps(sched, t)
    X = np.column_stack([M_traj[:-1] / M_SCALE, u_step])
    Y = (M_traj[1:] / M_SCALE)[:, None]
    return X, Y


def make_esn(units: int, sr: float, lr: float, ridge: float = 1e-6, seed: int = 42):
    res = Reservoir(
        units=units,
        lr=lr,
        sr=sr,
        input_scaling=0.5,
        rc_connectivity=0.1,
        seed=seed,
    )
    ro = Ridge(ridge=ridge)
    return res >> ro


def closed_loop_rollout(
    esn,
    M_true: np.ndarray,
    sched: Schedule,
    warmup_steps: int,
) -> np.ndarray:
    """Najpierw zasila rezerwuar prawdziwymi danymi przez `warmup_steps` kroków,
    a następnie przechodzi na tryb zamknięty (własna predykcja M_hat jako kolejne wejście)."""
    u_step = dose_steps(sched, t)
    esn.reset()
    M_hat = np.empty_like(M_true)
    M_hat[: warmup_steps + 1] = M_true[: warmup_steps + 1]

    for k in range(warmup_steps):
        xk = np.array([M_true[k] / M_SCALE, u_step[k]])
        esn.step(xk)

    m_curr = M_true[warmup_steps]
    for k in range(warmup_steps, len(t) - 1):
        xk = np.array([m_curr / M_SCALE, u_step[k]])
        y_next = float(np.asarray(esn.step(xk)).ravel()[0])
        m_next = max(0.0, y_next * M_SCALE)
        M_hat[k + 1] = m_next
        m_curr = m_next
    return M_hat


def rel_l2(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b) / np.linalg.norm(b))

## 1. Zbiór uczący, walidacyjny i testowy harmonogramów

Żeby sieć nauczyła się zarówno reakcji na pojedyncze frakcje o różnym odstępie, jak i swobodnego
odrostu po leczeniu, zestaw uczący zawiera 7 różnych harmonogramów (bez leczenia, krótkie i długie
serie o różnych odstępach i porach startu); **dane M(t) liczymy z PDE**. Hiperparametry wybieramy na dwóch
harmonogramach walidacyjnych, a dopiero potem oceniamy sieć na dwóch zupełnie niewidzianych harmonogramach testowych:
- **Test A (bazowy z #8):** 10 frakcji co $0{,}4$ od $t = 1{,}0$ (`Schedule.regular(10, 1.0, 0.4)`),
- **Test B (kurs dzielony z przerwą):** 5 frakcji od $t = 0{,}8$ co $0{,}35$, przerwa, a potem 5 frakcji od $t = 4{,}5$ co $0{,}35$,
- **Test C (ekstrapolacja w czasie):** sieć uczona wyłącznie na odcinku $[0, T_{\text{train}}]$ ($T_{\text{train}} = 5{,}5$),
  oceniana na $[T_{\text{train}}, 10{,}0]$ harmonogramu A.

In [ ]:
train_schedules = [
    NO_TREATMENT,
    Schedule.regular(n=4, start=1.0, interval=0.5),
    Schedule.regular(n=6, start=0.8, interval=0.6),
    Schedule.regular(n=8, start=1.2, interval=0.35),
    Schedule.regular(n=12, start=0.6, interval=0.3),
    Schedule.regular(n=7, start=2.0, interval=0.45),
    Schedule(times=(0.7, 1.1, 1.5, 3.0, 3.4, 3.8, 5.2, 5.6), width=0.05),
]
val_schedules = [
    Schedule.regular(n=9, start=1.5, interval=0.38),
    Schedule(times=(0.9, 1.3, 1.7, 2.1, 3.6, 4.0, 4.4), width=0.05),
]

sched_test_a = Schedule.regular(n=10, start=1.0, interval=0.4)
sched_test_b = Schedule(
    times=tuple(round(0.8 + i * 0.35, 4) for i in range(5))
    + tuple(round(4.5 + i * 0.35, 4) for i in range(5)),
    width=0.05,
)

# dane uczące i walidacyjne pochodzą z PDE
train_M = [pde_mass(sc) for sc in train_schedules]
X_train_list, Y_train_list = zip(*(build_xy(M_tr, sc) for M_tr, sc in zip(train_M, train_schedules)))
val_set = [(pde_mass(sc), sc) for sc in val_schedules]

M_pde_a = pde_mass(sched_test_a)
M_pde_b = pde_mass(sched_test_b)
M_ode_a = ode2_mass(sched_test_a)  # ręcznie dobrane ODE_2 z #20: punkt odniesienia
M_ode_b = ode2_mass(sched_test_b)

T_TRAIN = 5.5
idx_tr = int(np.argmin(np.abs(t - T_TRAIN)))

## 2. Siatka hiperparametrów (`units`, `sr`, `lr`, `ridge`)

Wynik konfiguracji to średnia z dwóch błędów względnych $L^2$ względem PDE na harmonogramach walidacyjnych:
1. **pełny rollout zamknięty** po krótkim rozgrzaniu rezerwuaru (`warmup_steps = 10`, czyli $t \in [0, 0{,}5]$),
2. **ekstrapolacja:** uczenie tylko na $[0, T_{\text{train}}]$ i prognoza na $[T_{\text{train}}, 10]$.

Testy A, B i C nie biorą udziału w doborze. Dodaliśmy `ridge`, bo przy słabej regularyzacji ekstrapolacja
potrafi się rozbiec (masa rośnie poza zakres uczący, a pętla zamknięta to wzmacnia).

In [ ]:
grid_rows = []
for units in (150, 300):
    for sr in (0.5, 0.9):
        for lr in (0.2, 0.5):
            for ridge in (1e-5, 1e-3, 1e-2):
                esn = make_esn(units=units, sr=sr, lr=lr, ridge=ridge, seed=42)
                esn.fit(list(X_train_list), list(Y_train_list), warmup=10)
                full = np.mean([rel_l2(closed_loop_rollout(esn, M, sc, warmup_steps=10), M) for M, sc in val_set])

                esn_e = make_esn(units=units, sr=sr, lr=lr, ridge=ridge, seed=42)
                esn_e.fit([X[:idx_tr] for X in X_train_list], [Y[:idx_tr] for Y in Y_train_list], warmup=10)
                ext = np.mean([
                    rel_l2(closed_loop_rollout(esn_e, M, sc, warmup_steps=idx_tr)[idx_tr:], M[idx_tr:])
                    for M, sc in val_set
                ])
                grid_rows.append({
                    "units": units, "sr": sr, "lr": lr, "ridge": ridge,
                    "L2_walid_rollout": round(full, 4),
                    "L2_walid_ekstrapolacja": round(ext, 4),
                    "wynik": round(0.5 * (full + ext), 4),
                })

df_grid = pd.DataFrame(grid_rows).sort_values("wynik").reset_index(drop=True)
df_grid.to_csv(TABLES / "zad3_02_esn_siatka.csv", index=False)
print("Top 8 konfiguracji (średni błąd walidacyjny vs PDE):")
print(df_grid.head(8).to_string(index=False))
print("\nNajgorsze 4 konfiguracje:")
print(df_grid.tail(4).to_string(index=False))

best = df_grid.iloc[0]
BEST_UNITS = int(best["units"])
BEST_SR = float(best["sr"])
BEST_LR = float(best["lr"])
BEST_RIDGE = float(best["ridge"])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8), sharex=True, sharey=True)
for ax, rg in zip(axes, (1e-5, 1e-3, 1e-2)):
    sub = df_grid[df_grid["ridge"] == rg].copy()
    sub["konfig"] = sub["units"].astype(str) + " / " + sub["sr"].astype(str)
    piv = sub.pivot(index="konfig", columns="lr", values="wynik").sort_index()
    im = ax.imshow(np.clip(piv.values, 0.0, 0.5), cmap="viridis_r", vmin=0.0, vmax=0.5, aspect="auto")
    ax.set_xticks(range(len(piv.columns)), [str(c) for c in piv.columns])
    ax.set_yticks(range(len(piv.index)), list(piv.index))
    ax.set_title(f"ridge = {rg:g}")
    ax.set_xlabel("leak rate (lr)")
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            lbl = f"{v:.3f}" if v < 1.0 else ">1.0"
            ax.text(j, i, lbl, ha="center", va="center", color="white" if v > 0.25 else "black", fontsize=9)
axes[0].set_ylabel("units / promień spektralny (sr)")
fig.colorbar(im, ax=axes, shrink=0.85, label="błąd walidacyjny vs PDE (ucięty do 0,5)")
fig.suptitle("Siatka hiperparametrów ESN uczonej na PDE (walidacja: rollout i ekstrapolacja)", y=1.02)
fig.savefig(FIGURES / "zad3_02_esn_siatka.png", dpi=150, bbox_inches="tight")
plt.close(fig)

## 3. Testy: nowe harmonogramy (A, B) i ekstrapolacja poza $[0, T_{\text{train}}]$ (C)

Trenujemy najlepszą konfigurację dwukrotnie:
- **ESN (nowe harmonogramy):** uczona na 7 harmonogramach treningowych (bez A i B), a na A i B rozwijana autonomicznie od $t = 0{,}5$ do $t = 10{,}0$.
- **ESN (ekstrapolacja):** uczona wyłącznie na przedziale $[0, T_{\text{train}}]$ ($T_{\text{train}} = 5{,}5$) tego samego zestawu uczącego
  i prefiksu trajektorii A z PDE, a następnie rozwijana autonomicznie na $[5{,}5,\,10{,}0]$.

In [ ]:
# Wariant 1: pełny rollout na zupełnie nowych harmonogramach A i B
esn_gen = make_esn(units=BEST_UNITS, sr=BEST_SR, lr=BEST_LR, ridge=BEST_RIDGE, seed=42)
esn_gen.fit(list(X_train_list), list(Y_train_list), warmup=10)
M_esn_a = closed_loop_rollout(esn_gen, M_pde_a, sched_test_a, warmup_steps=10)
M_esn_b = closed_loop_rollout(esn_gen, M_pde_b, sched_test_b, warmup_steps=10)

# Wariant 2: ekstrapolacja poza [0, T_train] na harmonogramie A (uczymy tylko na prefiksach z PDE)
Xa_full, Ya_full = build_xy(M_pde_a, sched_test_a)
X_ext = [X[:idx_tr] for X in X_train_list] + [Xa_full[:idx_tr]]
Y_ext = [Y[:idx_tr] for Y in Y_train_list] + [Ya_full[:idx_tr]]
esn_ext = make_esn(units=BEST_UNITS, sr=BEST_SR, lr=BEST_LR, ridge=BEST_RIDGE, seed=42)
esn_ext.fit(X_ext, Y_ext, warmup=10)
M_esn_ext = closed_loop_rollout(esn_ext, M_pde_a, sched_test_a, warmup_steps=idx_tr)

## 4. Tabela porównawcza: ESN (uczona na PDE) a `ODE_2` a PDE

Dla ESN dodatkowo sprawdzamy odporność na losowanie rezerwuaru: ta sama konfiguracja z pięcioma różnymi ziarnami.

In [ ]:
comp_rows = []
for label, M_pde, M_ode, M_esn, sl in [
    ("Test A (bazowy, nowy harmonogram)", M_pde_a, M_ode_a, M_esn_a, slice(None)),
    ("Test B (kurs dzielony, nowy harmonogram)", M_pde_b, M_ode_b, M_esn_b, slice(None)),
    ("Test C (ekstrapolacja t > 5,5 na A)", M_pde_a, M_ode_a, M_esn_ext, slice(idx_tr, None)),
]:
    mmin_p, _ = min_mass(t, M_pde)
    mmin_o, _ = min_mass(t, M_ode)
    mmin_e, _ = min_mass(t, M_esn)
    treg_p = time_to_regrowth(t, M_pde)
    treg_o = time_to_regrowth(t, M_ode)
    treg_e = time_to_regrowth(t, M_esn)

    comp_rows.append({
        "eksperyment": label,
        "L2_ESN_vs_PDE": round(rel_l2(M_esn[sl], M_pde[sl]), 4),
        "L2_ODE2_vs_PDE": round(rel_l2(M_ode[sl], M_pde[sl]), 4),
        "M_min_PDE": round(mmin_p, 3),
        "M_min_ODE2": round(mmin_o, 3),
        "M_min_ESN": round(mmin_e, 3),
        "t_odrostu_PDE": round(treg_p, 2) if np.isfinite(treg_p) else np.nan,
        "t_odrostu_ODE2": round(treg_o, 2) if np.isfinite(treg_o) else np.nan,
        "t_odrostu_ESN": round(treg_e, 2) if np.isfinite(treg_e) else np.nan,
    })

df_comp = pd.DataFrame(comp_rows)
df_comp.to_csv(TABLES / "zad3_02_esn_porownanie.csv", index=False)
print(f"Najlepsze hiperparametry: units={BEST_UNITS}, sr={BEST_SR}, lr={BEST_LR}, ridge={BEST_RIDGE:g}")
print(df_comp.to_string(index=False))

# odporność na ziarno rezerwuaru
seed_rows = []
for sd in range(5):
    e1 = make_esn(units=BEST_UNITS, sr=BEST_SR, lr=BEST_LR, ridge=BEST_RIDGE, seed=sd)
    e1.fit(list(X_train_list), list(Y_train_list), warmup=10)
    e2 = make_esn(units=BEST_UNITS, sr=BEST_SR, lr=BEST_LR, ridge=BEST_RIDGE, seed=sd)
    e2.fit(X_ext, Y_ext, warmup=10)
    pa = closed_loop_rollout(e1, M_pde_a, sched_test_a, warmup_steps=10)
    pb = closed_loop_rollout(e1, M_pde_b, sched_test_b, warmup_steps=10)
    pc = closed_loop_rollout(e2, M_pde_a, sched_test_a, warmup_steps=idx_tr)
    seed_rows.append({
        "ziarno": sd,
        "L2_testA": round(rel_l2(pa, M_pde_a), 4),
        "L2_testB": round(rel_l2(pb, M_pde_b), 4),
        "L2_testC": round(rel_l2(pc[idx_tr:], M_pde_a[idx_tr:]), 4),
    })
df_seed = pd.DataFrame(seed_rows)
df_seed.to_csv(TABLES / "zad3_02_esn_ziarna.csv", index=False)
print("\nOdporność na ziarno rezerwuaru:")
print(df_seed.to_string(index=False))
print(df_seed[["L2_testA", "L2_testB", "L2_testC"]].agg(["median", "max"]).round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.3), sharey=True)

panels = [
    (axes[0], "Test A: nowy harmonogram regularny", M_pde_a, M_ode_a, M_esn_a, sched_test_a, 0.5),
    (axes[1], "Test B: nowy kurs dzielony z przerwą", M_pde_b, M_ode_b, M_esn_b, sched_test_b, 0.5),
    (axes[2], f"Test C: ekstrapolacja poza $T_{{train}}={T_TRAIN:.1f}$", M_pde_a, M_ode_a, M_esn_ext, sched_test_a, T_TRAIN),
]

for ax, title, mp, mo, me, sc, t_split in panels:
    ax.axvspan(0.0, t_split, color="tab:gray", alpha=0.12, label="rozgrzewka / przedział uczący")
    ax.plot(t, mp, color="black", lw=2.2, label="PDE (CN)")
    ax.plot(t, mo, color="tab:orange", ls="--", lw=2.0, label="ODE_2 (ręczne, #20)")
    ax.plot(t, me, color="tab:green", lw=2.0, label="ESN uczona na PDE (closed-loop)")
    for ti in sc.times:
        ax.axvline(ti, color="tab:red", alpha=0.15, lw=0.8)
    ax.axhline(M0, color="gray", ls=":", lw=0.9)
    ax.set_title(title)
    ax.set_xlabel("czas $t$")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("masa guza $M(t)$")
axes[0].legend(frameon=False, fontsize=8.5, loc="upper left")
fig.suptitle(
    f"ESN uczona na PDE (units={BEST_UNITS}, sr={BEST_SR}, lr={BEST_LR}, ridge={BEST_RIDGE:g}) vs ODE_2 vs PDE — pole przesunięte",
    y=1.01,
)
fig.tight_layout()
fig.savefig(FIGURES / "zad3_02_esn_porownanie.png", dpi=150, bbox_inches="tight")
plt.close(fig)

## Wnioski

- **Sieć uczona na danych z PDE przewyższa ręczne `ODE_2`:** na nowych harmonogramach błąd względem PDE wynosi
  5,6% (test A) i 6,0% (test B), a `ODE_2` ma 22,7% i 25,5%. Sieć odtwarza też to, czego `ODE_2` nie ma:
  dalszy wzrost guza po t ≈ 7 (w `ODE_2` stała pojemność `K_eff` daje nasycenie), a w teście B dokładniej
  trafia czas do odrostu (3,95 wobec 5,65 w PDE; `ODE_2` ma 3,0).
- **Ekstrapolacja w czasie (test C):** po uczeniu tylko na $[0, 5{,}5]$ błąd na $[5{,}5, 10]$ to 2,8% wobec 23,2% dla `ODE_2`.
- **Odporność na ziarno rezerwuaru:** dla pięciu ziaren błąd to 4–5% (A), 8–13% (B) i 3–37% (C). Nowe harmonogramy są
  więc stabilne i zawsze lepsze niż `ODE_2`, a ekstrapolacja zależy od ziarna: dwa z pięciu ziaren (ok. 36–37%)
  są gorsze od `ODE_2`. Wynik testu C z ziarnem 42 jest więc optymistyczny, a medianą z ziaren jest ok. 12%.
- **Rola hiperparametrów:** `ridge` ma duże znaczenie. Przy `ridge = 1e-5` ekstrapolacja potrafiła się rozbiec
  (masa rosła poza zakres uczący), a `ridge = 1e-3` ją stabilizuje. Hiperparametry wybieramy na osobnych
  harmonogramach walidacyjnych, a nie na testowych.
- **Ograniczenia:** jedna geometria wiązki ($s = 6$, $r = 8$) i jedna wartość $\beta$; sieć nie była testowana
  na innych planach wiązki. Pętla zamknięta bez ostrzeżenia gubi dokładność, gdy $M$ wychodzi poza zakres uczący.
